# Syrin — Get Started

This notebook is **standalone**: install Syrin, set API keys and model names, then run every example. Each section is **independent** (creates its own agent/model).

**Covers:** Agent, Model, Context, Budget, Threshold, RateLimit, Memory, Checkpoints, Guardrails, Hooks, Tools, Loops, Multi-agent, Streaming, Prompts, **Serving** (HTTP & Web playground), and real-world budget + trace/debug.

---
## Install Syrin

Run the cell below.

In [ ]:
!pip install syrin

---
## Install Dependencies

Run the cell below to install provider SDKs and core deps. Leave as-is for Almock (no API key). Add your keys in the next section to use OpenAI or Anthropic.

In [ ]:
!pip install openai
!pip install anthropic
!pip install pydantic
!pip install httpx
!pip install tiktoken
!pip install python-dotenv

---
## Set API keys

Set your API keys below. Leave empty (`""`) to use **Almock** (no key required). The user handles environment as needed.

In [ ]:
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = ""

---
## Setup (imports and model helper)

Run after setting API keys. Defines `get_model()`: uses OpenAI if `OPENAI_API_KEY` is set, else Anthropic if `ANTHROPIC_API_KEY` is set, else Almock.

In [ ]:
import logging

logging.basicConfig(level=logging.ERROR)
for name in ("httpx", "httpcore", "asyncio"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

from syrin import Agent, Budget, Memory, MemoryType, Model, task, tool, warn_on_exceeded


def get_model():
    """OpenAI if OPENAI_API_KEY set, else Anthropic if ANTHROPIC_API_KEY set, else Almock."""
    if OPENAI_API_KEY:
        return Model.OpenAI("gpt-4o", api_key=OPENAI_API_KEY)
    if ANTHROPIC_API_KEY:
        return Model.Anthropic("claude-sonnet-4-5", api_key=ANTHROPIC_API_KEY)
    return Model.Almock(latency_min=0, latency_max=0, lorem_length=80)


model = get_model()
print(
    "Model:",
    "OpenAI" if OPENAI_API_KEY else "Anthropic" if ANTHROPIC_API_KEY else "Almock (no key)",
)

---
## Ways to create a model

Syrin supports multiple model backends. Use the one that matches your API key (or Almock for no key).

In [ ]:
from syrin import Model

# 1. Almock — no API key (mock responses for demos/tests)
m_almock = Model.Almock(latency_min=0, latency_max=0, lorem_length=40)

# 2. OpenAI — set OPENAI_API_KEY in the cell above
# m_openai = Model.OpenAI("gpt-4o", api_key=OPENAI_API_KEY)

# 3. Anthropic — set ANTHROPIC_API_KEY in the cell above
# m_anthropic = Model.Anthropic("claude-sonnet-4-5", api_key=ANTHROPIC_API_KEY)

# Use get_model() for automatic choice based on keys
m = get_model()
print("Active model type:", type(m).__name__)

---
## Ways to create an agent

You can define an agent in several ways: class-based, inline, or one-off with `syrin.run()`.

In [ ]:
from syrin import Agent, Model


# 1. Class-based — subclass Agent, set model and system_prompt as class attributes
class Helper(Agent):
    model = get_model()
    system_prompt = "You are a helpful assistant."


agent1 = Helper()

# 2. Inline — Agent(model=..., system_prompt=...) without subclassing
agent2 = Agent(model=get_model(), system_prompt="You answer briefly.")

# 3. One-off — syrin.run() for a single call (no agent instance)
import syrin

out = syrin.run("Say hi in 5 words.", model=get_model(), system_prompt="Be brief.")
print("Class-based:", agent1.response("2+2?").content[:50])
print("Inline:", agent2.response("2+2?").content[:50])
print("One-off:", out.content[:50])

---
## 1. Minimal — Basic agent

Create an agent with a model and system prompt; call `response()` and read `content`, `cost`, `tokens`.

In [ ]:
class Assistant(Agent):
    model = get_model()
    system_prompt = "You are a helpful assistant."


assistant = Assistant()
result = assistant.response("What is 2 + 2?")
print("Question: What is 2 + 2?")
print("Answer:", result.content)
print("Cost: $", result.cost)
print("Tokens:", result.tokens.total_tokens)

### Quick run — one-liner

`syrin.run()` for a single call without creating an agent.

In [ ]:
import syrin

result = syrin.run("What is Python?", model=get_model(), system_prompt="Explain in one sentence.")
print(result.content[:200])

### Agent with budget

Set a per-run cost limit and optional `on_exceeded` callback (`warn_on_exceeded` or `raise_on_exceeded`).

In [ ]:
class CostAwareAgent(Agent):
    model = get_model()
    system_prompt = "You are a concise assistant."
    budget = Budget(run=0.10, on_exceeded=warn_on_exceeded)


agent = CostAwareAgent()
r = agent.response("Explain quantum computing briefly")
print("Response:", r.content[:80], "...")
print("Budget state:", agent.budget_state)

### Agent with tools

Register tools with `@tool`; the agent can call them during `response()`.

In [ ]:
@tool
def calculate(a: float, b: float, operation: str = "add") -> str:
    """Perform arithmetic: add, subtract, multiply, divide."""
    if operation == "add":
        return str(a + b)
    if operation == "subtract":
        return str(a - b)
    if operation == "multiply":
        return str(a * b)
    if operation == "divide":
        return str(a / b) if b != 0 else "Error"
    return "Unknown"


class MathAgent(Agent):
    model = get_model()
    system_prompt = "Use tools for calculations."
    tools = [calculate]


agent = MathAgent()
result = agent.response("What is 15 times 7?")
print("Answer:", result.content)
print("Tool calls:", len(result.tool_calls))

### Agent — main fields (model, budget, memory, context, guardrails, loop, checkpoint, debug)

Agent supports: **model**, **system_prompt**, **tools**, **budget**, **memory**, **context**, **guardrails**, **loop**, **rate_limit**, **checkpoint**, **debug**, **events** (hooks).

In [ ]:
from syrin import AgentConfig
from syrin.checkpoint import CheckpointConfig, CheckpointTrigger
from syrin.context import Context

agent = Agent(
    model=get_model(),
    system_prompt="You are helpful.",
    tools=[],
    budget=Budget(run=0.50, on_exceeded=warn_on_exceeded),
    memory=Memory(),
    config=AgentConfig(
        context=Context(max_tokens=8000),
        checkpoint=CheckpointConfig(trigger=CheckpointTrigger.MANUAL),
    ),
    guardrails=None,
    loop=None,
    debug=False,
)
r = agent.response("Say hi briefly.")
print("Response:", r.content[:80], "...")
print("budget_state:", agent.budget_state)

---
## 2. Tasks — @syrin.task

Define named entry points with `@task`. Call via `agent.method_name()` or the task spec.

In [ ]:
class Researcher(Agent):
    model = get_model()
    system_prompt = "You are a research assistant. Provide concise summaries."

    @task
    def research(self, topic: str) -> str:
        """Research a topic and return a summary."""
        response = self.response(f"Research and summarize: {topic}")
        return response.content or ""


researcher = Researcher()
summary = researcher.research("AI in healthcare")
print("Topic: AI in healthcare")
print("Summary:", summary[:200], "...")

---
## 3. Budget — Limits and thresholds

**Basic:** `Budget(run=..., on_exceeded=...)`  
**Thresholds:** Fire a callback at a percentage of budget (e.g. 50%, 80%).

In [ ]:
from syrin.enums import ThresholdMetric
from syrin.threshold import BudgetThreshold, ThresholdContext

events = []


def on_50(ctx: ThresholdContext) -> None:
    events.append(f"50% threshold: {ctx.percentage}%")


agent = Agent(
    model=get_model(),
    budget=Budget(
        run=0.10,
        thresholds=[BudgetThreshold(at=50, action=on_50, metric=ThresholdMetric.COST)],
        on_exceeded=warn_on_exceeded,
    ),
)
agent.response("Hello!")
print("Threshold events:", events)

### RateLimit — Cost per hour/day (Budget.per)

`Budget(per=RateLimit(hour=..., day=...))` limits spend per time window. Use with `budget_store` for persistence across runs.

In [ ]:
from syrin import RateLimit

# Budget with per-run and per-hour rate limit (no store = in-memory only)
budget = Budget(run=1.0, per=RateLimit(hour=5.0, day=50.0), on_exceeded=warn_on_exceeded)
agent = Agent(model=get_model(), system_prompt="You are concise.", budget=budget)
r = agent.response("Hello")
print("Cost:", r.cost, "| Budget state:", agent.budget_state)

---
## 4. Memory — Remember & recall

Use `Memory()` for 4-type persistent memory; `remember()` and `recall()` with `MemoryType` (CORE, EPISODIC, SEMANTIC, PROCEDURAL).

In [ ]:
assistant = Agent(
    model=get_model(), system_prompt="You remember user preferences.", memory=Memory()
)
assistant.remember("The user's name is Alice.", memory_type=MemoryType.CORE, importance=1.0)
assistant.remember("User asked about ML.", memory_type=MemoryType.EPISODIC)
result = assistant.response("What's my name?")
print("Response:", result.content)

---
## 5. Tools — TOON format & structured output

Tools use TOON schema by default (fewer tokens). Use `Output(type)` and `@structured` for typed responses.

In [ ]:
# Simple tool with @tool
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get weather for a city."""
    return f"Weather in {city}: 22°{unit[0].upper()}"


agent = Agent(model=get_model(), system_prompt="Use tools when needed.", tools=[get_weather])
r = agent.response("What's the weather in Paris?")
print(r.content[:100])

In [ ]:
# (Context snapshot example is in Section 11.)

---
## 6. Loops — Single-shot vs ReAct

`SingleShotLoop` = one LLM call. `ReactLoop` = think → act (tools) → observe. `HumanInTheLoop` = approve tool calls.

In [ ]:
from syrin.loop import ReactLoop, SingleShotLoop

agent_ss = Agent(model=get_model(), loop=SingleShotLoop())
r1 = agent_ss.response("What is the capital of France?")
print("SingleShot:", r1.content[:80], "...")

agent_react = Agent(model=get_model(), loop=ReactLoop(max_iterations=5))
r2 = agent_react.response("What is 5 + 3?")
print("ReAct:", r2.content[:80], "...")

---
## 7. Multi-agent — Handoff & pipeline

**Handoff:** One agent passes work to another. **Pipeline:** Chain or parallel agents.

In [ ]:
class Analyzer(Agent):
    model = get_model()
    system_prompt = "You analyze and extract key findings."


class Presenter(Agent):
    model = get_model()
    system_prompt = "You present information clearly."


analyzer = Analyzer()
r1 = analyzer.response("Analyze benefits of renewable energy")
print("Analyzer:", r1.content[:80], "...")
r2 = analyzer.handoff(Presenter, "Present the analysis")
print("Presenter:", r2.content[:80], "...")

---
## 8. Streaming

`agent.stream(input)` yields chunks; use `chunk.text` and `chunk.cost_so_far`.

In [ ]:
agent = Agent(model=get_model(), system_prompt="You are helpful.")
chunks = list(agent.stream("Say hello in one sentence."))
full = "".join(c.text for c in chunks)
print("Streamed:", full)
print("Chunks:", len(chunks))

---
## 9. Guardrails

`ContentFilter`, `GuardrailChain`; attach to agent with `guardrails=`.

In [ ]:
from syrin.enums import GuardrailStage
from syrin.guardrails import ContentFilter, GuardrailChain

chain = GuardrailChain([ContentFilter(blocked_words=["spam", "scam"], name="NoSpam")])
print("Clean:", chain.check("Hello!", GuardrailStage.INPUT).passed)
print("Blocked:", chain.check("This is spam", GuardrailStage.INPUT).passed)

agent = Agent(model=get_model(), system_prompt="You are helpful.", guardrails=chain)
r = agent.response("Hello")
print("Agent:", r.content[:50], "...")

---
## 10. Observability — Hooks & debug

**Hooks:** `agent.events.on_response()`, `on_complete()`, `on_tool()` to track cost and behavior.  
**Debug:** `Agent(..., debug=True)` for automatic tracing (or use `--trace` when running scripts).

In [ ]:
total_cost = {"value": 0.0}


def track(ctx):
    total_cost["value"] += ctx.get("cost", 0)


agent = Agent(model=get_model(), system_prompt="You are helpful.")
agent.events.on_response(track)
agent.events.on_complete(lambda ctx: print("  Done. Cost: $", ctx.get("cost", 0)))
agent.response("Hello")
agent.response("How are you?")
print("Total cost:", total_cost["value"])

In [ ]:
# Same agent with debug=True — enables tracing (console/OTLP)
agent_debug = Agent(model=get_model(), system_prompt="You are helpful.", debug=True)
r = agent_debug.response("What is AI?")
print("Response:", r.content[:80], "...")

---
## 11. Context

`Context(max_tokens=..., thresholds=...)` for context-window and compaction. See **examples/11_context/** for demos: snapshot, thresholds, compaction, runtime inject, formation mode pull, output chunks, persistent map.

In [ ]:
from syrin.context import Context

ctx = Context(max_tokens=8000)
agent = Agent(model=get_model(), system_prompt="You are helpful.", context=ctx)
r = agent.response("Hello")
print("OK:", r.content[:50])

In [ ]:
# Context snapshot: full view of what was sent to the LLM (provenance, why_included, context_rot_risk, breakdown)
snap = agent.context.snapshot()
print(f"Utilization: {snap.utilization_pct:.1f}% | Context rot risk: {snap.context_rot_risk}")
if snap.breakdown:
    print("Breakdown:", snap.breakdown)
print("Why included:", snap.why_included[:5])
print("Provenance (first 3):", snap.provenance[:3] if snap.provenance else [])
print("Export for viz:", list(snap.to_dict().keys()))

---
## 12. Checkpoints

Save/load state with `save_checkpoint()`, `load_checkpoint()`, `list_checkpoints()`; configure triggers (MANUAL, STEP, TOOL, ERROR, BUDGET).

In [ ]:
agent = Agent(model=get_model(), system_prompt="You are a research assistant.")
cid = agent.save_checkpoint()
print("Saved checkpoint:", cid)
print("List:", agent.list_checkpoints())

---
## 13. Models

Use `get_model()` from the setup cell: it returns OpenAI or Anthropic if API keys are set, else Almock. Set OPENAI_MODEL / ANTHROPIC_MODEL and API keys in the second cell.

In [ ]:
import os

# get_model() returns OpenAI/Anthropic when keys are set (see setup cell).
print(
    "Active model:",
    "OpenAI"
    if os.environ.get("OPENAI_API_KEY")
    else "Anthropic"
    if os.environ.get("ANTHROPIC_API_KEY")
    else "Almock",
)

---
## 14. Prompts

`@prompt` decorator for reusable prompt functions.

In [ ]:
from syrin import prompt


@prompt
def assistant_prompt() -> str:
    return "You are a concise assistant. Answer in one or two sentences."


agent = Agent(model=get_model(), system_prompt=assistant_prompt())
r = agent.response("What is 2+2?")
print(r.content[:100])

---
## 15. Advanced — Config & inheritance

`syrin.configure(trace=True)`, `get_config()`. Define agents as classes with class-level `model`, `system_prompt`, `budget`.

In [ ]:
import syrin

syrin.configure(trace=False)
print("Trace:", syrin.get_config().trace)


class BaseAssistant(Agent):
    model = get_model()
    system_prompt = "You are helpful."


class ExpertAssistant(BaseAssistant):
    system_prompt = "You are an expert. Give detailed answers."


r = ExpertAssistant().response("What is Python?")
print("Expert:", r.content[:80], "...")

---
## 16. Serving — HTTP & Web playground

Serve your agent over HTTP with `agent.serve()`. Use `enable_playground=True` for a browser chat UI; `debug=True` for built-in observability (cost, tokens, traces per reply). Requires `pip install syrin[serve]`.

**In a notebook:** `serve()` blocks, so run the example scripts in a terminal to try the playground. Or uncomment the serve line below (will block until interrupted).

In [ ]:
# Create an agent (same as elsewhere in this notebook)
agent = Agent(
    model=get_model(),
    system_prompt="You are a helpful assistant. Be concise.",
    budget=Budget(run=0.5, on_exceeded=warn_on_exceeded),
)

# serve() exposes HTTP endpoints: POST /chat, POST /stream, GET /playground
# enable_playground=True → browser UI at http://localhost:8000/playground
# debug=True → observability panel (cost, tokens, traces per reply)
# agent.serve(port=8000, enable_playground=True, debug=True)

# Run in a terminal to try the playground:
#   pip install syrin[serve]
#   python -m examples.16_serving.playground_single
#   # Visit http://localhost:8000/playground

# Or run the full chatbot example:
#   python -m examples.16_serving.chatbot
#   # Chat with context, memory, guardrails, checkpoints

---
## Real-world: Budget + threshold + trace/debug

**Use case:** Run an agent with a tight budget; when a **threshold** is hit (e.g. 50% of budget), log or switch behavior. Use **debug=True** or **--trace** to see cost and spans.

1. Set a low run budget and a threshold at 50%.
2. Make a call; the threshold callback fires when spend crosses 50%.
3. Run with `debug=True` to see trace output.

In [ ]:
from syrin.enums import ThresholdMetric
from syrin.threshold import BudgetThreshold, ThresholdContext

threshold_hit = []


def on_threshold(ctx: ThresholdContext) -> None:
    threshold_hit.append({"pct": ctx.percentage, "cost": ctx.current_value})
    print(
        f"  [REAL-WORLD] Threshold hit: {ctx.percentage}% of budget used (${ctx.current_value:.6f})"
    )


agent = Agent(
    model=get_model(),
    system_prompt="You are concise.",
    budget=Budget(
        run=0.05,
        thresholds=[BudgetThreshold(at=50, action=on_threshold, metric=ThresholdMetric.COST)],
        on_exceeded=warn_on_exceeded,
    ),
    debug=True,  # Enable tracing — use --trace when running as script
)
result = agent.response("Explain machine learning in two sentences.")
print("Response:", result.content[:120], "...")
print("Budget state:", agent.budget_state)
print("Threshold events:", threshold_hit)